实验题目：全连接神经网络            

实验目的：掌握全连接神经网络用于回归和分类的工作流程

实验环境（硬件和软件）: Anaconda/Jupyter notebook/Pycharm

实验内容：

使用发布的“nihe.csv”数据集，使用前三列作为输入特征，最后一列作为回归标签，构建一个全连接神经网络，进行标量回归任务。画出训练集和验证集的损失变化曲线，以及测试集的真实值vs预测值曲线。

使用Keras框架下的MNIST手写数字数据集，搭建一个全连接神经网络，进行手写数字分类任务，画出训练集和验证集的损失变化曲线，以及对测试集的评估结果（准确率）。

CIFAR-10和CIFAR-100是来自于80 million张小型图片的数据集，图片收集者是Alex Krizhevsky, Vinod Nair, and Geoffrey Hinton。暂时先不管CIFAR-100数据集，以下是CIFAR-10数据集介绍：





Keras中，可以用load_data()函数进行加载，与MNIST数据集加载方式类似，具体加载示例如下：



实验要求：参考第2题中MNIST数据集的分类网络（全连接神经网络），在其基础上进行适当修改，使之能正确运行，从而进行图片分类任务。

使用CIFAR-10数据集，搭建一个卷积神经网络对图片进行分类（可参考PPT课件提供的案例），并与第3题中全连接神经网络结果进行对比。思考：从两种网络的参数量和准确率等方面对网络的性能进行综合评估。


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler  # 数据标准化
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# --- Re0 的““咒语””：设置中文显示 ---
# 确保你的环境中有 SimHei 字体 (黑体), 否则中文会显示为方框
# 如果你没有 SimHei，可以改成 'Microsoft YaHei' (微软雅黑)
font_list = [f.name for f in fm.fontManager.ttflist]
if 'SimHei' in font_list:
    plt.rcParams['font.sans-serif'] = ['SimHei']
else:
    print("警告：未找到 'SimHei' 字体，中文标签可能无法正确显示。")

plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

print(f"TensorFlow 版本: {tf.__version__}")
print("所有库导入成功！")

In [ ]:
# --- 1. 加载数据 ---
# 确保 "nihe.csv" 和你的 ipynb 文件在同一个文件夹下
try:
    data = pd.read_csv("nihe.csv")
except FileNotFoundError:
    print("--- 错误 ---")
    print("未找到 'nihe.csv'。请确保它和你的 Notebook 在同一目录下。")
    # 如果文件未找到，后续代码会出错，所以这里可以提前停止
    raise

# --- 2. 准备特征 (X) 和标签 (y) ---
# 任务要求：前三列为输入，最后一列为标签
X = data.iloc[:, :3]  # 选择前3列
y = data.iloc[:, -1] # 选择最后1列

# 打印数据信息，确认无误
print("--- 数据加载成功 ---")
print(f"总样本数: {len(data)}")
print("\n特征 (X) 的前5行:")
print(X.head())
print("\n标签 (y) 的前5行:")
print(y.head())

In [ ]:
# --- 3. 数据标准化 ---
# 对神经网络来说，将不同范围的特征缩放到同一个尺度非常重要
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# --- 4. 划分数据集 ---
# 60% 训练集, 20% 验证集, 20% 测试集
# (这是 Chap02 模型评估与选择.pptx 中讲到的““留出法””)

# 步骤一：先分出 60% 训练集 和 40% 临时集
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y,
    test_size=0.4,  # 40% 作为临时集
    random_state=42 # 设一个““随机种子””，保证每次划分结果都一样
)

# 步骤二：再把 40% 的临时集对半分，得到 20% 验证集和 20% 测试集
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5, # 0.5 * 40% = 20%
    random_state=42
)

print("\n--- 数据划分完毕 ---")
print(f"训练集 X 形状: {X_train.shape} | 训练集 y 形状: {y_train.shape}")
print(f"验证集 X 形状: {X_val.shape} | 验证集 y 形状: {y_val.shape}")
print(f"测试集 X 形状: {X_test.shape} | 测试集 y 形状: {y_test.shape}")

In [ ]:
# --- 5. 构建全连接神经网络 (FCNN) ---

model = keras.Sequential([
    # 输入层：input_shape 必须是特征的数量 (这里是3)
    # 我们用 64 个神经元，激活函数用 'relu'
    keras.layers.Dense(64, activation='relu', input_shape=[X_train.shape[1]]),

    # 隐藏层：再加一层 64 个神经元的
    keras.layers.Dense(64, activation='relu'),

    # 输出层：用于标量回归，我们只需要 1 个神经元
    # 并且**没有**激活函数 (或者说，激活函数是 'linear')
    # 因为我们的输出 (y) 是一个连续的数值，不是 0 或 1
    keras.layers.Dense(1)
])

# --- 6. 编译模型 ---
# 编译时，我们要告诉 Keras 三件事：
# 1. optimizer (优化器): 用什么方法去学习？ 'adam' 是最常用的。
# 2. loss (损失函数): 如何评价““猜错””的程度？
#    对于回归任务，我们用 'mean_squared_error' (均方误差, MSE)
#    (这对应《西瓜书》第3章的公式(3.3) 和 Chap02 PPT)
# 3. metrics (监控指标): 我们在训练时想看什么指标？ 'mae' (平均绝对误差) 更直观

model.compile(optimizer='adam',
              loss='mean_squared_error', # MSE
              metrics=['mae']) # Mean Absolute Error

print("\n--- 模型结构 ---")
model.summary()

In [ ]:
# --- 7. 训练模型 ---
print("\n--- 开始训练模型 ---")
# 我们训练 100 轮 (epochs)
# validation_data=(X_val, y_val) 告诉模型在每轮训练后，
# 都去验证集上““考一次试””，这样我们就能监控““过拟合””
history = model.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_val, y_val),
    verbose=1  # 打印每一轮的训练日志
)
print("--- 模型训练完成 ---")

# 将训练历史保存为 DataFrame，方便后续绘图
history_df = pd.DataFrame(history.history)
history_df['epoch'] = range(1, len(history_df) + 1)

In [ ]:
# --- 8. 绘制训练集和验证集的损失变化曲线 ---
# (实验任务1 要求)

plt.figure(figsize=(10, 6))
plt.plot(history_df['epoch'], history_df['loss'], label='训练集损失 (Training Loss)')
plt.plot(history_df['epoch'], history_df['val_loss'], label='验证集损失 (Validation Loss)')

plt.title('任务1: 训练集与验证集损失变化曲线 (Loss Curve)', fontsize=16)
plt.xlabel('Epochs (训练轮次)', fontsize=12)
plt.ylabel('Loss (Mean Squared Error)', fontsize=12)
plt.legend()
plt.grid(True)
plt.savefig('task1_loss_curve.png') # 保存图像到本地
plt.show()

print("已保存损失曲线: task1_loss_curve.png")

In [ ]:
# --- 9. 在““未见过””的测试集上评估 ---
print("\n--- 评估测试集 (Test Set) ---")
test_loss_mse, test_loss_mae = model.evaluate(X_test, y_test, verbose=0)
print(f"测试集 均方误差 (MSE): {test_loss_mse:.4f}")
print(f"测试集 平均绝对误差 (MAE): {test_loss_mae:.4f}")

# --- 10. 绘制 "真实值 vs 预测值" 曲线 ---
# (实验任务1 要求)

# (1) 获取模型在测试集上的预测值
y_pred = model.predict(X_test).flatten() # .flatten() 将 (n, 1) 变为 (n,)

# (2) 绘制散点图
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.6, label='预测点')
plt.xlabel('真实值 (True Values)', fontsize=12)
plt.ylabel('预测值 (Predicted Values)', fontsize=12)
plt.title('任务1: 测试集真实值 vs 预测值', fontsize=16)

# (3) 绘制 y=x 完美预测线
# 如果预测完美，所有点都应该在这条红线上
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r-', alpha=0.75, label='完美预测 (y=x)')

plt.legend()
plt.grid(True)
plt.axis('equal') # 保持x,y轴等比例
plt.gca().set_aspect('equal', adjustable='box')
plt.savefig('task1_true_vs_pred.png') # 保存图像
plt.show()

print("已保存真实值vs预测值曲线: task1_true_vs_pred.png")

In [ ]:
# --- 9. 在““未见过””的测试集上评估 ---
print("\n--- 评估测试集 (Test Set) ---")
test_loss_mse, test_loss_mae = model.evaluate(X_test, y_test, verbose=0)
print(f"测试集 均方误差 (MSE): {test_loss_mse:.4f}")
print(f"测试集 平均绝对误差 (MAE): {test_loss_mae:.4f}")

# --- 10. 绘制 "真实值 vs 预测值" 曲线 ---
# (实验任务1 要求)

# (1) 获取模型在测试集上的预测值
y_pred = model.predict(X_test).flatten() # .flatten() 将 (n, 1) 变为 (n,)

# (2) 绘制散点图
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.6, label='预测点')
plt.xlabel('真实值 (True Values)', fontsize=12)
plt.ylabel('预测值 (Predicted Values)', fontsize=12)
plt.title('任务1: 测试集真实值 vs 预测值', fontsize=16)

# (3) 绘制 y=x 完美预测线
# 如果预测完美，所有点都应该在这条红线上
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r-', alpha=0.75, label='完美预测 (y=x)')

plt.legend()
plt.grid(True)
plt.axis('equal') # 保持x,y轴等比例
plt.gca().set_aspect('equal', adjustable='box')
plt.savefig('task1_true_vs_pred.png') # 保存图像
plt.show()

print("已保存真实值vs预测值曲线: task1_true_vs_pred.png")